In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import os 
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
DATA_PATH = "dataset/"
os.listdir(DATA_PATH)

['sample_submission.csv', 'test.csv', 'test_labels.csv', 'train.csv']

In [3]:
TEST_PATH = os.path.join(DATA_PATH, "test.csv")
TEST_LABELS_PATH = os.path.join(DATA_PATH, "test.labels.csv")
SAMPLE_PATH = os.path.join(DATA_PATH, "sample_submission.csv")
TRAIN_PATH = os.path.join(DATA_PATH, "train.csv")
test_data = pd.read_csv(TEST_PATH)
train_data = pd.read_csv(TRAIN_PATH)
test_labels = pd.read_csv(TEST_LABELS_PATH) if os.path.exists(TEST_LABELS_PATH) else None

In [4]:
df = pd.read_csv('dataset/train.csv')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
from prepocessing.text_preprocessing import TextPreprocessor
from prepocessing.text_preprocessing import TextPreprocessorNonStopword
preprocessor = TextPreprocessor()
preprocessor_nonstopword = TextPreprocessorNonStopword()
df['clean_comment_text'] = df['comment_text'].apply(
    preprocessor.preprocess_text
)

df['nonstopword_comment_text'] = df['comment_text'].apply(
    preprocessor_nonstopword.preprocess_text_non_stopword
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading tokenizers/punkt/english.pickle: Package
[nltk_data]     'tokenizers/punkt/english.pickle' not found in index


Labelin Data

In [6]:
#definisi kolom di dataset
label_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# jika ada di dalam kolom itu nilai nya 1 pasti toxic
df['is_toxic'] = df[label_columns].max(axis=1)

X1 = df['clean_comment_text']
X2 = df['nonstopword_comment_text']
Y = df['is_toxic']

print("Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):")
print(Y.value_counts())

Distribusi Target Y (0 = Non-Toxic, 1 = Toxic):
is_toxic
0    143346
1     16225
Name: count, dtype: int64


TF-IDF

In [ ]:
from features.tfidf_extractor import TextVectorizer
from sklearn.model_selection import train_test_split

#pake stop word
X1_train, X1_test, Y_train, Y_test = train_test_split(X1, Y, test_size=0.2, random_state=42)

vectorizer = TextVectorizer(max_features=5000)
X1_train_tfidf = vectorizer.fit_transform(X1_train)
X1_test_tfidf = vectorizer.transform(X1_test)

print("Shape X1_train TF-IDF:", X1_train_tfidf.shape)

#non stopword
X2_train, X2_test, Y2_train, Y2_test = train_test_split(X2, Y, test_size=0.2, random_state=42)

vectorizer_nonstopword = TextVectorizer(max_features=5000)
X2_train_tfidf = vectorizer.fit_transform(X2_train)
X2_test_tfidf = vectorizer.transform(X2_test)

print("Shape X2_train TF-IDF:", X2_train_tfidf.shape)

Shape X1_train TF-IDF: (127656, 5000)
Shape X2_train TF-IDF: (127656, 5000)


LOGISTIC REGRRESSION (MODEL)

In [8]:
from models.logistic_model import ToxicClassifier
from models.logistic_model import ToxicClassifierNonStopword

classifier = ToxicClassifier()
classifier_nonstopword = ToxicClassifierNonStopword()

#stopword
classifier.train(X1_train_tfidf, Y_train)
#nonstopword
classifier_nonstopword.train(X2_train_tfidf, Y_train)

#stopword
Y_pred = classifier.predict(X1_test_tfidf)
#nonstopword
Y2_pred = classifier_nonstopword.predict(X2_test_tfidf)

#hasil
report1 = classifier.evaluate(Y_test, Y_pred)
report2 = classifier_nonstopword.evaluate(Y_test, Y2_pred)

print("STOP WORD")
print(report1)
print("NON STOP WORD")
print(report2)

STOP WORD
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.92      0.62      0.74      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.81      0.86     31915
weighted avg       0.95      0.96      0.95     31915

NON STOP WORD
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.92      0.63      0.75      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.81      0.86     31915
weighted avg       0.96      0.96      0.95     31915



In [9]:

words = vectorizer_nonstopword.get_feature_names() 
coeffs = classifier_nonstopword.get_coefficients() 

df_indikator = pd.DataFrame({'Kata': words, 'Bobot': coeffs})
bobot_hell = df_indikator[df_indikator["Kata"] == "hell"]
print("10 INDIKATOR KATA TOXIC")
print(df_indikator.sort_values(by='Bobot', ascending=False).head(10).to_string(index=False))
print(bobot_hell)

print("\n" + "="*50 + "\n")

df_error = pd.DataFrame({
    'Teks': X2_test,
    'Asli': Y_test,
    'Prediksi': Y2_pred
})

fp = df_error[(df_error['Asli'] == 0) & (df_error['Prediksi'] == 1)]
fn = df_error[(df_error['Asli'] == 1) & (df_error['Prediksi'] == 0)]

print("False Positif")
print(fp['Teks'].head(3).values)

print("False Negative")
print(fn['Teks'].head(3).values)

NotFittedError: Vocabulary not fitted or provided